In [8]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data exported from Week 5.
regression_df = pd.read_csv(
    "regression_data.csv",
    parse_dates=["hour_local"]
)

# All remaining columns are predictors.
core_features = regression_df.columns.drop(
    ["hour_local", "avg_departure_delay"]
).tolist()

print(f"Loaded {len(regression_df):,} rows and {len(core_features)} predictors.")

Loaded 28,499 rows and 23 predictors.


In [9]:
# Sort our data by date and time
data = regression_df.sort_values("hour_local")

# X contains the features we use to make predictions
X = data[core_features]

# y contains the variable we want to predict
y = data["avg_departure_delay"]

In [10]:
# Sort data chronologically
data = regression_df.sort_values("hour_local")

X = data[core_features]
y = data["avg_departure_delay"]

In [11]:
from sklearn.model_selection import train_test_split

#Split into training, validation, and testing data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

Below is an implementation of random forest regression that iterates through different values of estimators and maximum depths of a random forest to determine the r2-scores of a large range of models. While it was not necessary, I printed the r2 of each model just so I could visually see that the code was running.

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

r2_scores = []

for i in range (1, 101):
    for j in range (1, 9):
        forest_model = RandomForestRegressor(
            n_estimators=i,
            max_depth=j,
            random_state=42
        )

        # Train model
        forest_model.fit(X_train, y_train)

        # Make validation predictions
        forest_pred = forest_model.predict(X_val)

        #Compute the r2-scores of each model and store in an array
        r2 = r2_score(y_val, forest_pred)
        r2_scores.append(r2)
        print(r2)

0.023741090582082514
0.048596486785008164
0.0660591656892533
0.042770629429374374
-0.04801944721596452
-0.025253239888111123
-0.07402233562483529
-0.08485291301376163
0.023791730009780077
0.04516296700269029
0.05768690734607562
0.055148783481312136
0.043896583940241096
0.06037463806902088
-0.011913978858566932
-0.024744346707778586
0.02025949955891715
0.04151283760560276
0.058860043747310686
0.06747179104918044
0.07103691797319078
0.0854628238800309
0.05430971252292682
0.0032447262028068025
0.01684182108020582
0.03848845395088185
0.060357369766289914
0.06969036573109033
0.07811936555943866
0.09125873728664291
0.050945632053628676
-0.002860661398352393
0.018840925063260405
0.038360831592256894
0.05822773026128858
0.0659762389742734
0.07012458674460831
0.08262790361319283
0.045516165131140984
0.005121898598258845
0.019987021241121616
0.04133403779646605
0.06329316011773456
0.07142277998218138
0.0736255666354334
0.08964617266026553
0.0628850084857433
0.03582182672491174
0.0181478513896944

NameError: name 'np' is not defined

The code above ended in an error after going through all the models, but I did not rerun it, since it took about 45 minutes to run fully. The error was just that I needed to import numpy, so I mo

In [21]:
import numpy as np

#Find the best r2-score by finding the max value in the array
best_index = np.argmax(r2_scores)

#Find the number of estimators and max depth associated with this index
best_estimators = (best_index // 8) + 1
best_depth = (best_index % 8) + 1

print("Best R2:", r2_scores[best_index])
print("Best estimators:", best_estimators)
print("Best depth:", best_depth)

Best R2: 0.09848254180733318
Best estimators: 22
Best depth: 6


Therefore, it is known that the best-performing random forest model, when considering a number of estimators between 1 and 100 and a max depth between 1 and 8, is the model with 22 estimators that have a max-depth of 6. The r2 score of this model is 0.09848.

Below, this model is trained on a combination of the training and validation data so it can be tested against the linear models made before, including the ridge regression model that was found to be the best-performing:

In [22]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

best_forest_model = RandomForestRegressor(
    n_estimators=22,
    max_depth=6,
    random_state=42
)

# Train model
best_forest_model.fit(X_train_val, y_train_val)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",22
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at

Now it can be tested on testing data:

In [23]:
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

# Make validation predictions
forest_pred = forest_model.predict(X_test)

r2 = r2_score(y_test, forest_pred)
rmse = root_mean_squared_error(y_test, forest_pred)
mae = mean_absolute_error(y_test, forest_pred)

print(f"{"r2: "} {r2}")
print(f"{"RMSE: "} {rmse}")
print(f"{"MAE"} {mae}")

r2:  0.1660667287845945
RMSE:  27.88431031880346
MAE 12.806682016268523


This can be compared to the metrics of the best ridge regression model found before:
- r2: 0.09965
- RMSE: 28.973
- MAE: 13.650

It can also be compared to the MAE value from quantile regression, which was 12.428

The random forest model with 22 estimators and a max-depth of 6 performs better than the best regression model found before (ridge regression with 21 features) in all 3 metrics. Meanwhile, it performs slightly worse than the MAE for quantile regression. It is signifcant, though, that the r2 of the final random forest model is 0.166, compared to 0.09965 for ridge regression.